# Dwarpal: offline indexing on Colab (GPU)

Runs `scripts/index_cameras.py` on a Colab GPU for **cached-mode** cameras and returns
`data/cache/<camera>/` folders to copy back into your local repo. Use it when your local GPU is
too small or busy. The output is identical to `make index` run locally.

1. *Runtime → Change runtime type → GPU (T4 is enough).*
2. Upload your processed videos (`data/processed/<dataset>/*.mp4` + `manifest.json`) to Google Drive,
   e.g. `MyDrive/dwarpal/processed/`. Nothing else leaves your machine.
3. Run all cells, then download `dwarpal_cache.zip` and unzip it into the repo root.

In [ ]:
REPO_URL = "https://github.com/jay14090/dwarpal.git"   # your fork/clone URL
BRANCH = "main"
DRIVE_PROCESSED = "/content/drive/MyDrive/dwarpal/processed"
CAMERAS = []          # empty = every cached camera in config/cameras.yaml
WEIGHTS = "models/yolo26s.pt"

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%cd /content
!rm -rf dwarpal && git clone --depth 1 -b $BRANCH $REPO_URL dwarpal
%cd /content/dwarpal
!pip -q install uv && uv sync --quiet
!mkdir -p data && rm -rf data/processed && cp -r "$DRIVE_PROCESSED" data/processed && ls data/processed/*

In [ ]:
import shlex
cams = " ".join(shlex.quote(c) for c in CAMERAS)
!DWARPAL_DEVICE=cuda uv run python scripts/index_cameras.py {cams} --weights {WEIGHTS}

In [ ]:
!cd /content/dwarpal && zip -qr /content/dwarpal_cache.zip data/cache && ls -la /content/dwarpal_cache.zip
from google.colab import files
files.download("/content/dwarpal_cache.zip")